# 03 — Silver Data Processing

## NYC Taxi Demand Prediction | Microsoft Fabric

### Objectifs
- Explorer les fichiers Bronze (Taxi + Météo).
- Analyser les schémas et la qualité des données.
- Définir les règles de nettoyage.
- Appliquer les transformations avec PySpark.
- Stocker les résultats en Delta Lake.

### Tables Silver attendues
- silver.taxi_trips
- silver.weather_hourly

Période : janvier à décembre 2025.
Zones : 230, 161, 237, 186, 87.

In [ ]:

from pathlib import Path
import json
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import types as T

YEAR = 2025

TAXI_ZONE_IDS = [230, 161, 237, 186, 87]

BRONZE_ROOT = Path("/lakehouse/default/Files/bronze")

WEATHER_PATH = (
    BRONZE_ROOT
    / "weather/historical_forecast"
    / str(YEAR)
)

print("Spark version :", spark.version)
print("Bronze :", BRONZE_ROOT)
print("Météo :", WEATHER_PATH)


In [ ]:

assert BRONZE_ROOT.exists(), "Dossier Bronze introuvable"

files = [
    p for p in BRONZE_ROOT.rglob("*")
    if p.is_file() and not p.name.startswith(".")
]

inventory = pd.DataFrame([
    {
        "path": str(p),
        "format": p.suffix.lower(),
        "size_mb": round(p.stat().st_size / 1024**2, 2)
    }
    for p in files
])

if inventory.empty:
    print("Aucun fichier trouvé dans Bronze")
else:
    display(
        inventory.groupby("format", as_index=False).agg(
            n_files=("path", "count"),
            total_mb=("size_mb", "sum")
        )
    )
    display(inventory.head(20))


In [ ]:

# Identifier les Parquet Taxi 2025
taxi_files = sorted([
    str(p) for p in files
    if p.suffix.lower() == ".parquet"
    and "yellow_tripdata_2025-" in p.name.lower()
])

# Identifier les JSON météo 2025
weather_files = sorted([
    p for p in files
    if p.suffix.lower() == ".json"
    and WEATHER_PATH in p.parents
    and p.name.startswith("weather_forecast_2025-")
])

print("Fichiers Taxi :", len(taxi_files))
print("Fichiers météo :", len(weather_files))

print("\nExemple Taxi :")
print(taxi_files[0] if taxi_files else "Non trouvé")

print("\nExemple météo :")
print(weather_files[0] if weather_files else "Non trouvé")


In [ ]:

# Chemin Spark relatif au Lakehouse par défaut
TAXI_PATH = (
    "Files/bronze/taxi/2025/"
    "yellow_tripdata_2025-01.parquet"
)

# Lecture du fichier Parquet
df_taxi_sample = spark.read.parquet(TAXI_PATH)

print("=== SCHÉMA NYC TAXI ===")
df_taxi_sample.printSchema()

print("=== NOMBRE DE LIGNES ===")
print(df_taxi_sample.count())

print("=== COLONNES ===")
print(df_taxi_sample.columns)

print("=== APERÇU ===")
display(df_taxi_sample.limit(10))


In [ ]:

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, DoubleType

# Nombre total de lignes
total = df_taxi_sample.count()

# Construire une condition selon le type de colonne
null_exprs = []

for field in df_taxi_sample.schema.fields:
    c = field.name

    condition = F.col(c).isNull()

    # NaN uniquement pour les colonnes Float / Double
    if isinstance(field.dataType, (FloatType, DoubleType)):
        condition = condition | F.isnan(F.col(c))

    null_exprs.append(
        F.sum(
            F.when(condition, 1).otherwise(0)
        ).alias(c)
    )

# Calcul des valeurs manquantes
null_counts = (
    df_taxi_sample
    .select(null_exprs)
    .first()
    .asDict()
)

# Rapport de qualité
null_report = pd.DataFrame([
    {
        "column": c,
        "data_type": str(df_taxi_sample.schema[c].dataType),
        "null_count": int(v or 0),
        "null_percentage": round(
            100 * int(v or 0) / total, 2
        ) if total else 0
    }
    for c, v in null_counts.items()
])

display(
    null_report.sort_values(
        "null_percentage",
        ascending=False
    )
)


In [ ]:

print("=== STATISTIQUES NUMÉRIQUES ===")

display(
    df_taxi_sample.summary(
        "count", "min", "max", "mean",
        "stddev", "25%", "50%", "75%"
    )
)

print("=== DISTRIBUTION DES ZONES ===")

if "PULocationID" in df_taxi_sample.columns:
    display(
        df_taxi_sample
        .groupBy("PULocationID")
        .count()
        .orderBy(F.desc("count"))
        .limit(20)
    )

print("=== COLONNES TEMPORELLES ===")

date_columns = [
    c for c in df_taxi_sample.columns
    if "datetime" in c.lower() or "date" in c.lower()
]

for c in date_columns:
    df_taxi_sample.select(
        F.min(c).alias("min_date"),
        F.max(c).alias("max_date")
    ).show()


In [ ]:

assert weather_files, "Aucun fichier météo trouvé"

with open(weather_files[0], "r", encoding="utf-8") as f:
    weather_raw = json.load(f)

print("=== CLÉS PRINCIPALES ===")
print(list(weather_raw.keys()))

print("\n=== VARIABLES HORAIRES ===")
print(list(weather_raw["hourly"].keys()))

print("\n=== UNITÉS ===")
print(weather_raw.get("hourly_units", {}))

print("\n=== EXEMPLE DE VALEURS ===")
for key, values in weather_raw["hourly"].items():
    print(key, ":", values[:5])


In [ ]:

hourly = weather_raw["hourly"]

df_weather_pd = pd.DataFrame(hourly)

print("Dimensions :", df_weather_pd.shape)

print("\nValeurs NULL :")
display(df_weather_pd.isnull().sum().to_frame("null_count"))

print("\nStatistiques :")
display(df_weather_pd.describe(include="all"))

print("\nAperçu :")
display(df_weather_pd.head(10))


In [ ]:

import re

weather_inventory = []

for p in weather_files:
    with open(p, "r", encoding="utf-8") as f:
        raw = json.load(f)

    times = raw.get("hourly", {}).get("time", [])

    match = re.search(
        r"weather_forecast_2025-(\d{2})_", p.name
    )

    weather_inventory.append({
        "zone": p.parent.name,
        "month": int(match.group(1)) if match else None,
        "n_hours": len(times),
        "first_hour": times[0] if times else None,
        "last_hour": times[-1] if times else None
    })

df_coverage = pd.DataFrame(weather_inventory)

display(
    df_coverage.sort_values(["zone", "month"])
)

print("Total de fichiers :", len(df_coverage))

display(
    df_coverage.groupby("zone").agg(
        n_files=("month", "count"),
        total_hours=("n_hours", "sum")
    )
)


---
## Partie II — Nettoyage Silver des 12 Parquet Taxi

**Règles demandées :** enlever les lignes contenant au moins un `NULL`, les `NaN` numériques, les doublons exacts et les trajets avec `passenger_count <= 0`. Restreindre les pickups à l'année 2025 et aux zones 230, 161, 237, 186, 87. **Les fichiers Bronze ne changent pas.**

⚠️ Supprimer les lignes dès qu'une colonne *quelconque* est NULL est une politique stricte, susceptible d'éliminer de nombreux trajets valides. Le tableau d'audit quantifie les pertes. Pour une prédiction de demande, il pourra être pertinent de réviser cette politique après analyse.

In [ ]:
# 11 — Charger TOUS les fichiers mensuels ; chemin Spark relatif à OneLake
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from pathlib import Path
import pandas as pd

YEAR = 2025
TAXI_ZONE_IDS = [230, 161, 237, 186, 87]
TAXI_GLOB = "Files/bronze/taxi/2025/yellow_tripdata_2025-*.parquet"

# Contrôler les mois avant de traiter le dataset
months_found = sorted({int(Path(p).stem.split("-")[-1]) for p in taxi_files})
print("Mois trouvés :", months_found)
assert months_found == list(range(1, 13)), "12 mois attendus : vérifier l'ingestion Taxi Bronze"

df_taxi_bronze = (spark.read.option("mergeSchema", "true").parquet(TAXI_GLOB))
required_taxi = {"passenger_count", "PULocationID", "tpep_pickup_datetime"}
assert required_taxi.issubset(df_taxi_bronze.columns), f"Colonnes absentes : {required_taxi - set(df_taxi_bronze.columns)}"
print("Nombre de colonnes :", len(df_taxi_bronze.columns))
df_taxi_bronze.printSchema()

In [ ]:
# 12 — Nettoyage et audit séquentiel (ne pas modifier Bronze)
# Les counts déclenchent des passes Spark sur les fichiers : prévoir du temps.
audit_taxi = []

def audit_step(name, df, previous=None):
    count = df.count()
    audit_taxi.append({"etape": name, "lignes_restantes": count,
                       "lignes_supprimees": 0 if previous is None else previous - count})
    print(f"{name} : {count:,} lignes")
    return count

n0 = audit_step("Bronze initial", df_taxi_bronze)

# NULL sur toutes les colonnes, conformément à la règle choisie
df_t1 = df_taxi_bronze.dropna(how="any")
n1 = audit_step("Sans NULL", df_t1, n0)

# NaN uniquement pour FloatType et DoubleType
float_cols = [f.name for f in df_t1.schema.fields
              if isinstance(f.dataType, (T.FloatType, T.DoubleType))]
valid_numbers = F.lit(True)
for c in float_cols:
    valid_numbers = valid_numbers & ~F.isnan(F.col(c))
df_t2 = df_t1.filter(valid_numbers)
n2 = audit_step("Sans NaN", df_t2, n1)

# Un doublon exact = toutes les colonnes identiques
df_t3 = df_t2.dropDuplicates()
n3 = audit_step("Sans doublons exacts", df_t3, n2)

# Supprimer passenger_count = 0, négatif ou invalide
df_t4 = df_t3.filter(F.col("passenger_count") > 0)
n4 = audit_step("Passagers > 0", df_t4, n3)

# Date 2025 (pickup) ; timestamps NYC conservés en heure locale telle que source
df_t5 = df_t4.filter(
    (F.col("tpep_pickup_datetime") >= F.lit("2025-01-01")) &
    (F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
)
n5 = audit_step("Pickups en 2025", df_t5, n4)

df_taxi_silver = df_t5.filter(F.col("PULocationID").isin(TAXI_ZONE_IDS))
n6 = audit_step("5 zones sélectionnées", df_taxi_silver, n5)

In [ ]:
# 13 — Synthèse des suppressions Taxi
report_taxi = pd.DataFrame(audit_taxi)
report_taxi["%_conserve_vs_bronze"] = (
    (100 * report_taxi["lignes_restantes"] / n0).round(2) if n0 else 0
)
display(report_taxi)
assert n6 > 0, "Aucun trajet après nettoyage : vérifier les critères avant toute écriture"

display(df_taxi_silver.groupBy("PULocationID").count().orderBy("PULocationID"))
df_taxi_silver.select(
    F.min("passenger_count").alias("min_passengers"),
    F.min("tpep_pickup_datetime").alias("first_pickup"),
    F.max("tpep_pickup_datetime").alias("last_pickup")
).show(truncate=False)

## Partie III — Nettoyage météo (60 JSON, 5 zones × 12 mois)

La météo Bronze est composée de réponses JSON Open-Meteo avec des listes `hourly`. Nous les aplatissons en une ligne par zone et heure. Toutes les colonnes météo sont conservées ; les lignes comportant un `NULL`/`NaN` sont supprimées, les doublons détectés et dédupliqués. **Attention** : les horodatages `hourly.time` sont en heure locale `America/New_York`, sans offset dans ces réponses ; la répétition d'une heure lors du passage à l'heure d'hiver ne peut pas être correctement désambiguïsée avec ce seul champ. La déduplication par zone+heure est donc une décision simplificatrice, à revoir avant une étude chronologique de production.

In [ ]:
# 14 — Lire les 60 fichiers JSON Bronze et aplatir les tableaux horaires
import json
import re
from pathlib import Path
from pyspark.sql import types as T

WEATHER_BASE = Path("/lakehouse/default/Files/bronze/weather/historical_forecast/2025")
ZONE_BY_SLUG = {
    "times_square_theatre_district": 230,
    "midtown_center": 161,
    "upper_east_side_south": 237,
    "penn_station_madison_sq_west": 186,
    "financial_district_north": 87,
}
WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "rain", "snowfall", "wind_speed_10m", "weather_code"
]

all_weather_files = sorted(WEATHER_BASE.glob("*/weather_forecast_2025-*.json"))
assert len(all_weather_files) == 60, f"60 JSON attendus, {len(all_weather_files)} détectés"
weather_rows = []
coverage = set()
for fp in all_weather_files:
    slug = fp.parent.name
    assert slug in ZONE_BY_SLUG, f"Zone inconnue : {slug}"
    match = re.fullmatch(r"weather_forecast_2025-(\d{2})_(.+)\.json", fp.name)
    assert match and match.group(2) == slug, f"Fichier inattendu : {fp}"
    month = int(match.group(1))
    coverage.add((slug, month))
    with fp.open(encoding="utf-8") as f:
        raw = json.load(f)
    hourly = raw.get("hourly") or {}
    times = hourly.get("time") or []
    assert times, f"Horaires absents : {fp}"
    assert all(v in hourly and len(hourly[v]) == len(times) for v in WEATHER_VARS), f"Variables incomplètes : {fp}"
    for i, timestamp in enumerate(times):
        weather_rows.append((
            ZONE_BY_SLUG[slug], slug, timestamp,
            *[float(hourly[v][i]) if hourly[v][i] is not None else None
              for v in WEATHER_VARS],
            fp.name
        ))
assert coverage == {(s, m) for s in ZONE_BY_SLUG for m in range(1, 13)}, "Couverture 5 zones x 12 mois incomplète"

schema_weather = T.StructType([
    T.StructField("taxi_zone_id", T.IntegerType(), False),
    T.StructField("zone_slug", T.StringType(), False),
    T.StructField("time_local", T.StringType(), True),
    *[T.StructField(v, T.DoubleType(), True) for v in WEATHER_VARS],
    T.StructField("source_file", T.StringType(), False),
])
df_weather_raw = spark.createDataFrame(weather_rows, schema=schema_weather)
print("Lignes météo initiales :", df_weather_raw.count())
df_weather_raw.printSchema()

In [ ]:
# 15 — Normaliser le temps, éliminer NULL/NaN et doublons météo
weather_audit = []
def weather_step(name, df, previous=None):
    n = df.count()
    weather_audit.append({"etape": name, "lignes_restantes": n,
                          "lignes_supprimees": 0 if previous is None else previous - n})
    print(name, n)
    return n

w0 = weather_step("Météo brute", df_weather_raw)

# Heure locale sans conversion implicite en UTC
# to_timestamp_ntz disponible avec Spark 3.4+
df_w1 = (df_weather_raw
    .withColumn("weather_hour_local", F.to_timestamp_ntz(F.col("time_local")))
    .drop("time_local"))
w1 = weather_step("Horodatage parsé", df_w1, w0)

df_w2 = df_w1.dropna(how="any")
w2 = weather_step("Sans NULL", df_w2, w1)

condition = F.lit(True)
for c in WEATHER_VARS:
    condition = condition & (~F.isnan(F.col(c)))
df_w3 = df_w2.filter(condition)
w3 = weather_step("Sans NaN", df_w3, w2)

# Unicité de (zone, heure) : conserver une occurrence par clé.
# Les fichiers Bronze sont immuables ; ici on résout les doublons avec un ordre stable.
from pyspark.sql import Window
window = Window.partitionBy("taxi_zone_id", "weather_hour_local").orderBy("source_file")
df_w4 = (df_w3.withColumn("_rank", F.row_number().over(window))
         .filter(F.col("_rank") == 1).drop("_rank"))
w4 = weather_step("Sans doublons zone/heure", df_w4, w3)

df_weather_silver = df_w4.filter(
    (F.col("weather_hour_local") >= F.lit("2025-01-01")) &
    (F.col("weather_hour_local") < F.lit("2026-01-01")) &
    F.col("taxi_zone_id").isin(TAXI_ZONE_IDS)
)
w5 = weather_step("2025 + 5 zones", df_weather_silver, w4)

report_weather = pd.DataFrame(weather_audit)
report_weather["%_conserve_vs_bronze"] = (100 * report_weather["lignes_restantes"] / w0).round(2)
display(report_weather)
display(df_weather_silver.groupBy("taxi_zone_id").count().orderBy("taxi_zone_id"))
assert w5 > 0, "Aucune ligne météo après nettoyage"

## Partie IV — Validation et écriture Delta Silver

Les cellules ci-dessous écrivent **deux tables séparées** : `silver.taxi_trips` et `silver.weather_hourly`. Aucun rapprochement Taxi × Météo n'est fait à cette étape ; la jointure et les variables ML appartiennent à Gold.

**Précondition :** relire les deux rapports d'audit. Le mode `overwrite` remplace les éventuelles tables Silver existantes. Nécessite un Lakehouse **avec schémas activés**, attaché comme Lakehouse par défaut.

In [ ]:
# 16 — Contrôles de validation AVANT l'écriture
assert df_taxi_silver.filter(F.col("passenger_count") <= 0).limit(1).count() == 0
assert df_taxi_silver.filter(~F.col("PULocationID").isin(TAXI_ZONE_IDS)).limit(1).count() == 0
assert df_weather_silver.filter(~F.col("taxi_zone_id").isin(TAXI_ZONE_IDS)).limit(1).count() == 0
assert df_weather_silver.groupBy("taxi_zone_id", "weather_hour_local").count().filter("count > 1").limit(1).count() == 0

print("Contrôles Silver réussis.")
print("Taxi :", n6, "lignes | Météo :", w5, "lignes")
print("IMPORTANT : vérifier les rapports et pertes de lignes avant l'écriture Delta.")

In [ ]:
# 17 — Écriture Delta Lake (EXÉCUTER SEULEMENT APRÈS VALIDATION)
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

(df_taxi_silver.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver.taxi_trips"))

(df_weather_silver.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("silver.weather_hourly"))

print("Tables Delta créées : silver.taxi_trips et silver.weather_hourly")

In [ ]:
# 18 — Vérifier les tables Delta persistées
for tab in ["silver.taxi_trips", "silver.weather_hourly"]:
    print("===", tab, "===")
    spark.sql(f"DESCRIBE DETAIL {tab}").select("format", "location").show(truncate=False)
    spark.table(tab).groupBy().count().show()

print("Bronze inchangé ; Silver prêt pour la couche Gold.")